<a href="https://colab.research.google.com/github/zynx0286-max/zynroblox/blob/main/SFX_Studio_Roblox_Edition_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Autonomous AI Audio Studio — Roblox Edition (100% Free, v5)

Fully local, free, mobile-safe, self-mastering, crash-resistant audio pipeline for Roblox creators.

**Cell order:** 1 Install → 2 Core Engine → 3 *(optional but recommended)* Mount Google Drive → 4 Music + Script Analyzer → 5 Video/Image Matcher → 6 Launch.

**New in v5 — reliability for big batches:**
- **Google Drive persistence (Cell 3, optional):** if you're generating a big batch (like a 100+ item script), run this cell and approve the Drive prompt. Every batch job now auto-checkpoints to Drive every 10 items, so a Colab disconnect mid-run no longer means lost work — just re-run and your progress is sitting in Drive already.
- **Live time estimates** on every batch job, so you know roughly how long a big run will take before it surprises you.
- **A "❓ Help" tab** (first tab in the app) that explains what every other tab does in one place.
- Every tab now has a one-line description so nothing is a mystery.

**About Colab and crashes:** the "reconnecting" message you saw is Google Colab's free-tier idle/session limit kicking in — not a bug in the app. See the note in the Help tab (and my explanation in chat) for exactly why, and what actually fixes it (hint: it's not really about 24/7 hosting — see below).

No API keys anywhere in this notebook.

## Cell 1 — Install dependencies
First run: ~4-6 min (downloads TangoFlux + MusicGen weights on first use).

In [ ]:
%%capture
# --- Installation Start --- #
# Clean up potentially problematic packages first to ensure a fresh install
!pip uninstall -y diffusers transformers tangoflux accelerate || echo "No existing packages to uninstall or failed to uninstall some (this is normal if they weren't installed)."

!apt-get -qq install -y ffmpeg libsndfile1

# Install core HuggingFace packages, ensuring they are upgraded to the latest compatible versions.
# AutoPipelineForTextToAudio should be present in the latest diffusers.
# Transformers and accelerate are often tightly coupled with diffusers, so installing the latest of each.
!pip install --force-reinstall diffusers transformers accelerate || echo "⚠️ Core HuggingFace packages install/upgrade failed. Please re-run this cell, or check your network settings if it keeps failing."

# Install TangoFlux with --no-deps to prevent it from pulling older, conflicting versions of diffusers, transformers, or accelerate.
!pip install --no-deps git+https://github.com/declare-lab/TangoFlux || echo "⚠️ TangoFlux install failed. Please re-run this cell, or check your network settings if it keeps failing."

# Manually install TangoFlux's other non-conflicting dependencies.
# If 'datasets' related errors occur, adjust accordingly.
!pip install --force-reinstall safetensors einops omegaconf torchlibrosa torchaudio || echo "⚠️ One or more TangoFlux support packages failed. TangoFlux generation may error later."

# Install everything else required by the notebook.
!pip install gradio soundfile librosa pydub scipy matplotlib pillow pyloudnorm

# --- Installation End --- #

import torch, transformers, gradio
print(f"✅ Dependencies installed.")
print(f"   torch={torch.__version__}  |  transformers={transformers.__version__}  |  gradio={gradio.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("   ⚠️ No GPU detected — go to Runtime → Change runtime type → T4 GPU, then re-run this cell.")

In [ ]:
# This cell's functionality has been moved to cell `uQ5VY1CP7krk` for a more streamlined installation process.
# Please restart the runtime and run all cells from the beginning after any modifications to `uQ5VY1CP7krk`.

In [ ]:
# This cell's functionality has been moved to cell `uQ5VY1CP7krk` for a more streamlined installation process.
# Forcibly reinstalling packages and requesting a restart is now handled comprehensively in the initial setup.
# Please restart the runtime and run all cells from the beginning after any modifications to `uQ5VY1CP7krk`.

## Cell 2 — Core engine: SFX generation, CLAP scoring, DSP, looping, diagnostics, export

In [ ]:
import os
import gc
import re
import math
import time
import shutil
import zipfile
import numpy as np
import scipy.signal as signal
import soundfile as sf
import librosa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr
import torch

from pydub import AudioSegment
from transformers import AutoProcessor, ClapModel
import pyloudnorm as pyln

try:
    from tangoflux import TangoFluxInference
    HAS_TANGOFLUX = True
except ImportError:
    HAS_TANGOFLUX = False

WORKSPACE = "./sfx_studio_workspace"
os.makedirs(f"{WORKSPACE}/exports", exist_ok=True)
os.makedirs(f"{WORKSPACE}/batch", exist_ok=True)

# Where batch checkpoints get saved. Defaults to local (ephemeral, wiped on
# disconnect) until/unless the optional Cell 3 (Mount Google Drive) is run,
# which repoints this at a Drive folder that survives disconnects.
PERSIST_DIR = f"{WORKSPACE}/exports"
DRIVE_MOUNTED = False
CHECKPOINT_EVERY = 10  # save progress to PERSIST_DIR every N items in a batch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU for fast generation.")


class ModelRegistry:
    tangoflux = None
    clap_model = None
    clap_processor = None
    musicgen_model = None
    musicgen_processor = None
    blip_model = None
    blip_processor = None


def _filepath(f):
    """Safely extract a filesystem path from a Gradio file-like value across
    Gradio versions (some return a plain string path, others a tempfile-like
    object with a .name attribute)."""
    if f is None:
        return None
    if isinstance(f, str):
        return f
    if hasattr(f, "name"):
        return f.name
    return None


def free_gpu_memory():
    """Unloads every loaded model so the next generation reloads fresh.
    Use this when switching between heavy tabs (SFX/Music/Video) if you hit
    out-of-memory errors on the free T4 GPU."""
    for attr in ["tangoflux", "clap_model", "clap_processor", "musicgen_model",
                 "musicgen_processor", "blip_model", "blip_processor"]:
        setattr(ModelRegistry, attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return "🧹 Cleared all loaded models from memory. The next generation will take a few extra seconds to reload whatever it needs."


def load_clap_evaluator():
    if ModelRegistry.clap_model is None:
        model_id = "laion/clap-htsat-fused"
        ModelRegistry.clap_processor = AutoProcessor.from_pretrained(model_id)
        ModelRegistry.clap_model = ClapModel.from_pretrained(model_id).to(DEVICE)
        ModelRegistry.clap_model.eval()


def load_tangoflux_engine():
    if ModelRegistry.tangoflux is None:
        if not HAS_TANGOFLUX:
            raise ImportError("TangoFlux library not found. Re-run Cell 1 to install it.")
        ModelRegistry.tangoflux = TangoFluxInference(name="declare-lab/TangoFlux", device=DEVICE)


def calculate_clap_score(text_prompt: str, audio_array: np.ndarray, sample_rate: int) -> float:
    load_clap_evaluator()
    if sample_rate != 48000:
        audio_48k = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=48000)
    else:
        audio_48k = audio_array
    if audio_48k.ndim > 1:
        audio_48k = np.mean(audio_48k, axis=0)

    inputs = ModelRegistry.clap_processor(
        text=[text_prompt], audio=[audio_48k], return_tensors="pt", # Changed 'audios' to 'audio'
        sampling_rate=48000, padding=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = ModelRegistry.clap_model(**inputs)
        score = outputs.logits_per_audio[0][0].cpu().item()
    return float(score)


def _generate_sfx_core(prompt: str, duration: float, steps: int, candidate_count: int, score_threshold):
    """Reusable core used by both the UI button and the Script Analyzer batch runner."""
    load_tangoflux_engine()
    best_audio = None
    best_score = -1.0
    logs = []

    duration = max(1.0, min(10.0, float(duration)))  # SFX hard-capped at 1-10s
    n = max(1, int(candidate_count))

    for i in range(n):
        generated_tensor = ModelRegistry.tangoflux.generate(prompt=prompt, steps=int(steps), duration=duration)
        audio_np = generated_tensor.cpu().numpy().squeeze() if isinstance(generated_tensor, torch.Tensor) else np.array(generated_tensor).squeeze()
        sr = 44100
        score = calculate_clap_score(prompt, audio_np, sr)
        logs.append(f"Candidate {i+1}: CLAP={score:.4f}")
        if score > best_score:
            best_score = score
            best_audio = (audio_np, sr)

    out_path = f"{WORKSPACE}/gen_raw_{os.urandom(4).hex()}.wav"
    audio_data, sr = best_audio
    sf.write(out_path, audio_data if audio_data.ndim == 1 else audio_data.T, sr)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out_path, best_score, "\n".join(logs)


def generate_sound_effect(prompt: str, duration: float, steps: int, candidate_count: int, score_threshold: float,
                           progress=gr.Progress()):
    if not prompt.strip():
        return None, "Error: Prompt string cannot be empty.", "N/A"
    try:
        progress(0.1, desc="Generating raw sound effect candidates...")
        raw_audio_path, best_score, log_summary = _generate_sfx_core(prompt, duration, steps, candidate_count, score_threshold)

        if best_score < score_threshold:
            sfx_status_msg = f"⚠️ Best SFX candidate score ({best_score:.4f}) below threshold ({score_threshold}). Kept anyway.\n" + log_summary
        else:
            sfx_status_msg = f"✅ Selected best SFX candidate (CLAP: {best_score:.4f})\n" + log_summary

        progress(0.7, desc="Applying automatic mastering...")
        processed_audio_path, auto_master_explanation = auto_master(raw_audio_path, progress=progress)

        if processed_audio_path is None:
            return None, f"Auto-Mastering Error: {auto_master_explanation}\n{sfx_status_msg}", "N/A"

        final_status_msg = sfx_status_msg + "\n\n" + auto_master_explanation
        return processed_audio_path, final_status_msg, f"{best_score:.4f}"
    except Exception as e:
        return None, f"Generation Error: {str(e)}", "N/A"


def process_dsp_chain(audio_path, hp_cutoff, eq_mid_gain, drive, reverb_mix, normalize_lufs, noise_gate_threshold_db=-60, high_shelf_gain=0, comp_threshold_db=0, comp_ratio=1.0, limiter_threshold_db=-0.5):
    if not audio_path or not os.path.exists(audio_path):
        return None, "Error: Invalid input audio path."
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=False)
        if y.ndim == 1:
            y = np.vstack([y, y])

        # Noise Gate
        if noise_gate_threshold_db < 0:
            threshold_amp = 10**(noise_gate_threshold_db / 20.0)
            # Apply gate to both channels
            for ch in range(2):
                mask = np.abs(y[ch]) < threshold_amp
                y[ch][mask] *= 0.1 # Attenuate by 10x, not fully silence to avoid clicks

        if hp_cutoff > 20:
            b_hp, a_hp = signal.butter(2, hp_cutoff / (sr / 2.0), btype="highpass")
            y[0] = signal.lfilter(b_hp, a_hp, y[0])
            y[1] = signal.lfilter(b_hp, a_hp, y[1])

        if eq_mid_gain != 0:
            b_mid, a_mid = signal.iirpeak(2000 / (sr / 2.0), Q=1.0)
            mid_boost = 10 ** (eq_mid_gain / 20.0)
            y[0] += signal.lfilter(b_mid, a_mid, y[0]) * (mid_boost - 1.0)
            y[1] += signal.lfilter(b_mid, a_mid, y[1]) * (mid_boost - 1.0)

        # High Shelf EQ
        if high_shelf_gain != 0:
            # A simple shelf filter for frequencies above ~8kHz
            # Using a basic first-order filter here. For more precision, consider iirfilter.
            b_hs, a_hs = signal.butter(1, 8000 / (sr / 2.0), btype='highpass')
            y_high = signal.lfilter(b_hs, a_hs, y)
            y += y_high * (10**(high_shelf_gain / 20.0) - 1.0)

        if drive > 0:
            gain = 1.0 + (drive * 4.0)
            y = np.tanh(y * gain) / np.tanh(gain)

        # Dynamic Range Compression (simple implementation)
        if comp_ratio > 1.0:
            threshold_amp_comp = 10**(comp_threshold_db / 20.0)
            for ch in range(2):
                for i in range(len(y[ch])):
                    if abs(y[ch][i]) > threshold_amp_comp:
                        y[ch][i] = threshold_amp_comp + (y[ch][i] - threshold_amp_comp) / comp_ratio

        if reverb_mix > 0:
            delay_samples = int(sr * 0.04)
            decay = reverb_mix * 0.5
            for ch in range(2):
                buf = np.copy(y[ch])
                for i in range(delay_samples, len(buf)): # Removed the trailing 'c'
                    buf[i] += buf[i - delay_samples] * decay
                y[ch] = (1 - reverb_mix * 0.3) * y[ch] + (reverb_mix * 0.3) * buf

        if normalize_lufs:
            try:
                meter = pyln.Meter(sr)
                current_lufs = meter.integrated_loudness(y.T)
                if math.isfinite(current_lufs):
                    gain_db = max(-24.0, min(24.0, -14.0 - current_lufs))
                    y = y * (10 ** (gain_db / 20.0))
                else:
                    raise ValueError
            except Exception:
                rms = np.sqrt(np.mean(y ** 2))
                if rms > 0:
                    y = y * ((10 ** (-14.0 / 20.0)) / rms)

        # Limiter (after LUFS normalization or other processing to prevent clipping)
        if limiter_threshold_db < 0:
            limit_amp = 10**(limiter_threshold_db / 20.0)
            y = np.clip(y, -limit_amp, limit_amp)

        processed_path = f"{WORKSPACE}/dsp_processed_{os.urandom(4).hex()}.wav"
        sf.write(processed_path, y.T, sr)
        return processed_path, "DSP processing completed successfully."
    except Exception as e:
        return None, f"DSP Processing Error: {str(e)}"


def generate_seamless_loop(audio_path, crossfade_ms):
    if not audio_path or not os.path.exists(audio_path):
        return None, "Error: Invalid audio file for looping."
    try:
        seg = AudioSegment.from_file(audio_path)
        xf_ms = int(crossfade_ms)
        if len(seg) <= xf_ms * 2:
            return None, "Error: Input audio duration too short for requested crossfade."
        head = seg[:-xf_ms]
        tail = seg[-xf_ms:]
        looped_seg = head.append(tail, crossfade=xf_ms)
        preview_3x = looped_seg * 3
        loop_out_path = f"{WORKSPACE}/loop_master_{os.urandom(4).hex()}.wav"
        preview_3x.export(loop_out_path, format="wav")
        return loop_out_path, f"Seamless loop synthesized ({crossfade_ms}ms crossfade)."
    except Exception as e:
        return None, f"Looping Error: {str(e)}"


def analyze_signal_metrics(audio_path):
    if not audio_path or not os.path.exists(audio_path):
        return "No audio loaded.", None
    try:
        y, sr = librosa.load(audio_path, sr=None)
        duration = len(y) / sr
        peak_val = float(np.max(np.abs(y)))
        rms_val = float(np.sqrt(np.mean(y ** 2)))
        peak_db = 20 * math.log10(peak_val) if peak_val > 0 else -100.0
        rms_db = 20 * math.log10(rms_val) if rms_val > 0 else -100.0

        try:
            meter = pyln.Meter(sr)
            measured_lufs = meter.integrated_loudness(y)
            if not math.isfinite(measured_lufs):
                raise ValueError
            lufs_label = "Integrated Loudness (ITU-R BS.1770)"
        except Exception:
            measured_lufs = rms_db - 3.1
            lufs_label = "Estimated Loudness (clip too short for full LUFS gating)"

        metrics_text = (f"Duration: {duration:.2f} s\nSample Rate: {sr} Hz\n"
                         f"Peak Level: {peak_db:.2f} dBFS\nRMS Level: {rms_db:.2f} dBFS\n"
                         f"{lufs_label}: {measured_lufs:.2f} LUFS")

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 4), gridspec_kw={'height_ratios': [1, 1.2]})
        fig.patch.set_facecolor('#0f172a')
        times = np.linspace(0, duration, len(y))
        ax1.set_facecolor('#1e293b')
        ax1.plot(times, y, color='#0284c7', alpha=0.8, linewidth=0.7)
        ax1.set_title('Waveform Display', color='#f8fafc', fontsize=9)
        ax1.tick_params(colors='#94a3b8', labelsize=8)
        ax1.grid(True, color='#334155', linestyle='--', alpha=0.5)

        stft_db = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        ax2.set_facecolor('#1e293b')
        ax2.imshow(stft_db, origin='lower', aspect='auto', cmap='magma', extent=[0, duration, 0, sr // 2])
        ax2.set_title('Frequency Spectrogram (Hz)', color='#f8fafc', fontsize=9)
        ax2.tick_params(colors='#94a3b8', labelsize=8)
        ax2.set_xlabel('Time (s)', color='#94a3b8', fontsize=8)
        plt.tight_layout()
        plot_path = f"{WORKSPACE}/diag_{os.urandom(4).hex()}.png"
        plt.savefig(plot_path, facecolor=fig.get_facecolor(), dpi=120)
        plt.close(fig)
        return metrics_text, plot_path
    except Exception as e:
        return f"Diagnostic Error: {str(e)}", None


def _export_one(audio_path, export_profile, out_dir=None):
    """Core exporter, returns output filepath. out_dir lets batch mode write into a temp folder."""
    out_dir = out_dir or f"{WORKSPACE}/exports"
    os.makedirs(out_dir, exist_ok=True)
    seg = AudioSegment.from_file(audio_path)
    out_name = f"sfx_master_{os.urandom(3).hex()}"

    if export_profile == "Roblox Mobile-Optimized (Mono OGG, low-mem)":
        # Tuned to be small + light on mobile CPU/RAM: mono, 44.1kHz, lower vorbis quality (~q2 ≈ 96kbps)
        seg = seg.set_frame_rate(44100).set_channels(1)
        out_file = f"{out_dir}/{out_name}_mobile.ogg"
        seg.export(out_file, format="ogg", codec="libvorbis", parameters=["-q:a", "2"])
    elif export_profile == "Roblox Mono OGG (44.1kHz, full quality)":
        seg = seg.set_frame_rate(44100).set_channels(1)
        out_file = f"{out_dir}/{out_name}.ogg"
        seg.export(out_file, format="ogg", codec="libvorbis")
    elif export_profile == "Unreal/Unity Stereo WAV (48kHz)":
        seg = seg.set_frame_rate(48000).set_channels(2)
        out_file = f"{out_dir}/{out_name}.wav"
        seg.export(out_file, format="wav")
    elif export_profile == "High-Quality Lossless FLAC":
        seg = seg.set_frame_rate(48000)
        out_file = f"{out_dir}/{out_name}.flac"
        seg.export(out_file, format="flac")
    else:
        seg = seg.set_frame_rate(44100)
        out_file = f"{out_dir}/{out_name}.mp3"
        seg.export(out_file, format="mp3", bitrate="192k")
    return out_file


def export_game_preset(audio_path, export_profile):
    if not audio_path or not os.path.exists(audio_path):
        return None, "Error: No audio asset selected for export."
    try:
        out_file = _export_one(audio_path, export_profile)
        size_kb = os.path.getsize(out_file) / 1024
        note = ""
        if export_profile.startswith("Roblox") and size_kb > 6800:
            note = " ⚠️ File is close to/over Roblox's ~7MB free-tier audio asset limit."
        return out_file, f"Export completed: {export_profile}. Size: {size_kb:.0f} KB.{note}"
    except Exception as e:
        return None, f"Export Error: {str(e)}"


def zip_files(filepaths, zip_name):
    """Zips a list of files. Only actually needed when >2 files, but safe to call always."""
    os.makedirs(f"{WORKSPACE}/batch", exist_ok=True)
    zip_path = f"{WORKSPACE}/batch/{zip_name}_{os.urandom(3).hex()}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in filepaths:
            if fp and os.path.exists(fp):
                zf.write(fp, arcname=os.path.basename(fp))
    return zip_path


def save_checkpoint(generated_files, batch_name):
    """Called periodically during long batch runs (every CHECKPOINT_EVERY items).
    Copies everything generated so far into PERSIST_DIR — Google Drive if Cell 3
    was run, otherwise this session's local storage — as both loose files and a
    rolling zip. If Colab disconnects mid-batch, whatever was already checkpointed
    is safe (on Drive, permanently; locally, only until the session is wiped)."""
    if not generated_files:
        return
    try:
        dest_dir = f"{PERSIST_DIR}/{batch_name}_checkpoint"
        os.makedirs(dest_dir, exist_ok=True)
        for fp in generated_files:
            if fp and os.path.exists(fp):
                dest = os.path.join(dest_dir, os.path.basename(fp))
                if not os.path.exists(dest):
                    shutil.copy2(fp, dest)
        zip_path = f"{PERSIST_DIR}/{batch_name}_checkpoint.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in generated_files:
                if fp and os.path.exists(fp):
                    zf.write(fp, arcname=os.path.basename(fp))
    except Exception as e:
        print(f"Checkpoint save skipped due to error: {e}")


def set_persist_dir(path, mounted):
    global PERSIST_DIR, DRIVE_MOUNTED
    PERSIST_DIR = path
    DRIVE_MOUNTED = mounted


print("✅ Cell 2 loaded: SFX engine, DSP, looping, diagnostics, export.")


# ---------------- Auto-Master (one-click automatic mastering) ----------------

def auto_master(audio_path, progress=gr.Progress()):
    """Analyzes the audio (loudness, spectral balance, crest factor) and picks
    its own EQ/saturation/loudness settings instead of you tuning sliders."""
    if not audio_path or not os.path.exists(audio_path):
        return None, "Error: No source audio provided."
    try:
        y, sr = librosa.load(audio_path, sr=None, mono=True)

        try:
            meter = pyln.Meter(sr)
            current_lufs = meter.integrated_loudness(y)
            if not math.isfinite(current_lufs):
                raise ValueError
        except Exception:
            current_lufs = -23.0

        stft = np.abs(librosa.stft(y))
        freqs = librosa.fft_frequencies(sr=sr)

        # Analyze for Mid and High presence
        presence_band = (freqs >= 2000) & (freqs <= 4000)
        presence_energy = stft[presence_band, :].sum()
        high_freq_band = (freqs >= 8000) & (freqs <= sr/2)
        high_freq_energy = stft[high_freq_band, :].sum()
        total_energy = stft.sum() + 1e-9

        presence_ratio = presence_energy / total_energy
        high_freq_ratio = high_freq_energy / total_energy

        peak = np.max(np.abs(y)) + 1e-9
        rms = np.sqrt(np.mean(y ** 2)) + 1e-9
        crest_db = 20 * math.log10(peak / rms)

        # Dynamic parameter setting
        eq_mid = 3 if presence_ratio < 0.08 else 0 # Boost mids if lacking presence
        high_shelf = 2 if high_freq_ratio < 0.05 and crest_db > 10 else 0 # Boost highs if lacking brightness and dynamic
        drive = 0.05 if crest_db < 12 else 0.0 # Gentle saturation for less dynamic sounds
        reverb = 0.12 # Light reverb for space
        hp_cutoff = 120 if high_freq_ratio < 0.03 else 80 # More aggressive HPF if sound lacks high-end (to clean up)

        # Noise gate threshold (simple estimate: slightly above the quietest RMS parts)
        frame_length = sr // 10 # 100ms frames
        rms_frames = []
        for i in range(0, len(y) - frame_length, frame_length):
            frame = y[i:i+frame_length]
            rms_frames.append(np.sqrt(np.mean(frame**2)))
        rms_frames = np.array(rms_frames)
        noise_gate_threshold_db = -60 # Default if no clear noise floor
        if len(rms_frames) > 0:
            # Estimate noise floor by a low percentile of RMS values, then add a small buffer
            noise_floor_rms = np.percentile(rms_frames, 10)
            if noise_floor_rms > 1e-6: # Avoid log of zero
                noise_gate_threshold_db = max(-60, 20 * math.log10(noise_floor_rms * 1.5)) # Ensure it's not too high

        # Compression parameters
        comp_threshold_db = -18.0 if crest_db > 16 else 0.0 # Compress if very dynamic
        comp_ratio = 3.0 if crest_db > 16 else 1.0
        limiter_threshold_db = -0.5 # Always apply a brickwall limiter to prevent clipping

        progress(0.5, desc="Applying automatic mastering chain...")
        out_path, _ = process_dsp_chain(audio_path, hp_cutoff, eq_mid, drive, reverb, True,
                                        noise_gate_threshold_db, high_shelf,
                                        comp_threshold_db, comp_ratio, limiter_threshold_db)

        explanation = (
            f"🧠 Auto-Master analysis:\n"
            f"  Input loudness: {current_lufs:.1f} LUFS -> normalized to -14 LUFS / -0.5 dBFS peak\n"
            f"  HPF: {hp_cutoff}Hz to remove rumble and mud.\n"
            f"  Mid Boost (2kHz): {eq_mid}dB ({'applied for clarity' if eq_mid else 'skipped (already balanced)'})\n"
            f"  High Shelf (8kHz+): {high_shelf}dB ({'applied for crispness' if high_shelf else 'skipped (already bright)'})\n"
            f"  Crest factor: {crest_db:.1f} dB -> saturation: "
            f"{'0.05, gentle warmth' if drive else 'skipped (transient sound, saturation would blunt attack)'}\n"
            f"  Noise Gate Threshold: {noise_gate_threshold_db:.1f} dB (to remove subtle background noise).\n"
            f"  Compression: Threshold {comp_threshold_db:.1f}dB, Ratio {comp_ratio:.1f}:1 ({'applied for dynamics control' if comp_ratio > 1 else 'skipped (dynamic enough)'})\n"
            f"  Limiter: -0.5dBFS (to prevent clipping and maximize loudness).\n"
            f"  Also applied: light reverb (0.12) for a sense of space."
        )
        return out_path, explanation
    except Exception as e:
        return None, f"Auto-Master Error: {str(e)}"


# ---------------- SFX Presets ----------------

SFX_PRESETS = {
    "🐉 Dragon": "Terrifying dragon roar echoing in a stone cavern, deep and powerful",
    "👹 Monster": "Guttural monster growl, threatening and low-pitched",
    "🧟 Zombie": "Zombie groan and shuffling footsteps, decayed and raspy",
    "🤖 Robot": "8-bit robotic beep with whirring servo motor",
    "🔮 Magic": "Shimmering magic spell being cast, sparkling and ethereal",
    "🕳️ Cave": "Cave echo ambience with dripping water and distant wind",
    "👣 Footsteps": "Footsteps on gravel, crunchy and rhythmic",
    "💥 Explosion": "Heavy explosion blast with debris and rumble",
    "🖱️ UI Click": "Clean UI button click, short and crisp",
    "⚔️ Sword Clash": "Metal sword clash, sharp ringing impact",
}


def apply_preset(preset_name):
    return SFX_PRESETS.get(preset_name, "")


# ---------------- Batch SFX from a prompt list ----------------

def generate_batch_sfx(prompt_lines, duration, steps, candidates, threshold, progress=gr.Progress()):
    if not prompt_lines or not prompt_lines.strip():
        return None, "Paste one prompt per line first."
    lines = [l.strip() for l in prompt_lines.splitlines() if l.strip()][:100]
    if not lines:
        return None, "No valid prompts found."

    drive_warning = ""
    if len(lines) > 20 and not DRIVE_MOUNTED:
        drive_warning = ("⚠️ Large batch without Google Drive mounted — if Colab disconnects mid-run, "
                          "unsaved progress will be lost. Consider running the optional Cell 3 (Mount Google Drive) first, "
                          "then coming back to this.\n\n")

    generated = []
    start_time = time.time()
    for i in range(len(lines)):
        p = lines[i]
        elapsed = time.time() - start_time
        avg = (elapsed / i) if i > 0 else 15.0
        remaining_min = (avg * (len(lines) - i)) / 60.0
        progress(i / len(lines), desc=f"{i+1}/{len(lines)}: {p[:35]}... (~{remaining_min:.1f} min left)")
        try:
            out_path, _, _ = _generate_sfx_core(p, duration, steps, candidates, threshold)
            # Auto-mastering is now done within _generate_sfx_core's call to generate_sound_effect
            # We need to make sure the batch generation uses the auto-mastered output.
            # The _generate_sfx_core is directly used here, so we need to apply auto_master explicitly.
            processed_out_path, _ = auto_master(out_path)
            exported = _export_one(processed_out_path, "Roblox Mobile-Optimized (Mono OGG, low-mem)", out_dir=f"{WORKSPACE}/batch")
            generated.append(exported)
        except Exception as e:
            print(f"Skipped '{p}': {e}")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(generated, "sfx_batch")

    save_checkpoint(generated, "sfx_batch")

    if not generated:
        return None, drive_warning + "All generations failed — check the log output below the cell."

    save_note = (f"Also safely saved to Google Drive at {PERSIST_DIR}/sfx_batch_checkpoint/"
                 if DRIVE_MOUNTED else
                 "Saved to this session's temporary storage only — download it now, it won't survive a disconnect.")
    if len(generated) > 2:
        zp = zip_files(generated, "batch_sfx")
        print(f"DEBUG: Zip file path: {zp}, exists: {os.path.exists(zp)}") # Added debug print here
        return zp, f"{drive_warning}✅ Generated {len(generated)} SFX and zipped them. {save_note}"
    else:
        return generated[0], f"{drive_warning}✅ Generated {len(generated)} SFX. {save_note}"


# ---------------- Session Library ----------------

def add_to_library(new_output, history):
    """Chained after generation buttons via .then() to auto-collect every
    output this session into a running library."""
    if history is None:
        history = []
    if new_output:
        if isinstance(new_output, (list, tuple)):
            for item in new_output:
                if item and os.path.exists(item) and item not in history:
                    history.append(item)
        elif os.path.exists(new_output) and new_output not in history:
            history.append(new_output)
    return history, history


def zip_session_library(history):
    if not history:
        return None, "Session library is empty — generate something first!"
    valid = [h for h in history if h and os.path.exists(h)]
    if len(valid) > 2:
        zp = zip_files(valid, "session_library")
        return zp, f"📦 Zipped {len(valid)} files from this session."
    return None, f"You have {len(valid)} file(s) so far — zipping kicks in automatically once you pass 2. Download individually from the list above for now."


def clear_session_library():
    return [], []


print("✅ Cell 2 loaded: SFX engine, DSP (+ real LUFS + Auto-Master), looping, diagnostics, export, presets, batch mode, session library.")

In [ ]:
import sys
import importlib
import os

# Diagnostic: Check currently loaded diffusers version/path BEFORE attempting import
print("-- DEBUG: diffusers module diagnostic ---")
_diffusers_path = None
if 'diffusers' in sys.modules:
    try:
        # Importing here ensures we get the module object for reload
        import diffusers as _diffusers_diag
        _diffusers_path = _diffusers_diag.__path__[0] if hasattr(_diffusers_diag, '__path__') else None
        print(f"DEBUG: diffusers version already in sys.modules: {_diffusers_diag.__version__}")
        print(f"DEBUG: diffusers path already in sys.modules: {_diffusers_path}")
        print("DEBUG: diffusers module already loaded.")
    except Exception as e:
        print(f"DEBUG: Error checking diffusers from sys.modules: {e}")
else:
    print("DEBUG: diffusers module not found in sys.modules before explicit import.")

# --- Additional File System Checks ---
if _diffusers_path and os.path.exists(_diffusers_path):
    print(f"DEBUG: Listing contents of {_diffusers_path}:")
    try:
        for item in sorted(os.listdir(_diffusers_path)):
            print(f"  - {item}")
        pipelines_dir = os.path.join(_diffusers_path, 'pipelines')
        if os.path.exists(pipelines_dir) and os.path.isdir(pipelines_dir):
            print(f"DEBUG: 'pipelines' directory exists. Listing its contents:")
            for item in sorted(os.listdir(pipelines_dir)):
                print(f"  - pipelines/{item}")
        else:
            print("DEBUG: 'pipelines' directory NOT found within diffusers installation.")
    except Exception as e:
        print(f"DEBUG: Error listing diffusers directory contents: {e}")
else:
    print("DEBUG: diffusers installation path not found or invalid for file system check.")

# Now attempt the main import, using the specific path identified in diagnostics
# AutoPipelineForTextToAudio is expected to be in diffusers.pipelines.auto_pipeline
from diffusers.pipelines.auto_pipeline import AutoPipelineForText2Audio

# After the import, verify the version again to be certain
import diffusers # Re-import to ensure we get the (potentially reloaded) module reference
print(f"DEBUG: diffusers version AFTER explicit import: {diffusers.__version__}")
print(f"DEBUG: diffusers path AFTER explicit import: {diffusers.__path__}")
print("--- END DEBUG ---")

# Extend ModelRegistry for new models
class ModelRegistry:
    tangoflux = None
    clap_model = None
    clap_processor = None
    musicgen_model = None
    musicgen_processor = None
    blip_model = None
    blip_processor = None
    audiogen_model = None
    stable_audio_model = None

# Update free_gpu_memory to include new models
def free_gpu_memory():
    """Unloads every loaded model so the next generation reloads fresh.
    Use this when switching between heavy tabs (SFX/Music/Video) if you hit
    out-of-memory errors on the free T4 GPU."""
    for attr in ["tangoflux", "clap_model", "clap_processor", "musicgen_model",
                 "musicgen_processor", "blip_model", "blip_processor",
                 "audiogen_model", "stable_audio_model"]:
        setattr(ModelRegistry, attr, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return "🧹 Cleared all loaded models from memory. The next generation will take a few extra seconds to reload whatever it needs."


def load_audiogen_engine():
    if ModelRegistry.audiogen_model is None:
        # Using AutoPipelineForTextToAudio for easier integration
        ModelRegistry.audiogen_model = AutoPipelineForText2Audio.from_pretrained(
            "facebook/audiogen-medium", torch_dtype=torch.float16, device=DEVICE
        ).to(DEVICE)
        ModelRegistry.audiogen_model.eval()

def load_stable_audio_engine():
    if ModelRegistry.stable_audio_model is None:
        # Use AutoPipelineForTextToAudio for Stable Audio model
        ModelRegistry.stable_audio_model = AutoPipelineForText2Audio.from_pretrained(
            "stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16
        ).to(DEVICE)
        ModelRegistry.stable_audio_model.eval()


def select_sfx_model(prompt: str):
    prompt_lower = prompt.lower()
    if any(keyword in prompt_lower for keyword in [
        "foley", "realistic", "weapon", "gun", "impact", "explosion", "footstep", "environmental"
    ]):
        return "audiogen"
    elif any(keyword in prompt_lower for keyword in [
        "ambient", "texture", "atmosphere", "horror", "space", "drone"
    ]):
        return "stable_audio"
    else:
        return "tangoflux" # Default to TangoFlux


def _generate_sfx_with_model(model_name: str, prompt: str, duration: float, steps: int, candidate_count: int):
    """General purpose SFX generation that dispatches to the selected model."""
    best_audio = None
    best_score = -1.0
    logs = []
    sr = 44100 # Default sample rate for TangoFlux, might need adjustment for others

    if model_name == "audiogen":
        load_audiogen_engine()
        # Audiogen generates longer, so we generate once and potentially trim/loop
        # duration here is generally limited by the model
        audio_gen = ModelRegistry.audiogen_model(prompt,
                                             forward_params={"do_sample": True, "num_inference_steps": steps},
                                             audio_length_in_s=duration).audios[0]
        sr = ModelRegistry.audiogen_model.scheduler.config.sample_rate # Get actual SR
        audio_np = audio_gen
        # For simpler scoring, we'll just take the first candidate generated by audiogen/stable_audio
        score = calculate_clap_score(prompt, audio_np, sr)
        best_audio = (audio_np, sr)
        best_score = score
        logs.append(f"Audiogen candidate: CLAP={score:.4f}")

    elif model_name == "stable_audio":
        load_stable_audio_engine()
        # Stable Audio generates fixed length, often 30 seconds. Trim if necessary.
        # AutoPipelineForTextToAudio will handle duration if the model supports it.
        # The `duration` parameter should be passed directly to the pipeline call.
        pipeline_output = ModelRegistry.stable_audio_model(prompt,
                                                num_inference_steps=steps,
                                                num_waveforms_per_prompt=candidate_count,
                                                duration=duration)
        audio_gen = pipeline_output.audios
        sr = ModelRegistry.stable_audio_model.config.sampling_rate # Get actual SR from pipeline config
        audio_np = audio_gen[0] # Take first candidate
        # Ensure audio is trimmed to the exact duration if necessary, as Stable Audio might produce slightly longer outputs
        target_samples = int(duration * sr)
        if len(audio_np) > target_samples:
            audio_np = audio_np[:target_samples]

        score = calculate_clap_score(prompt, audio_np, sr)
        best_audio = (audio_np, sr)
        best_score = score
        logs.append(f"Stable Audio candidate: CLAP={score:.4f}")

    elif model_name == "tangoflux":
        load_tangoflux_engine()
        for i in range(candidate_count):
            generated_tensor = ModelRegistry.tangoflux.generate(prompt=prompt, steps=int(steps), duration=duration)
            audio_np = generated_tensor.cpu().numpy().squeeze() if isinstance(generated_tensor, torch.Tensor) else np.array(generated_tensor).squeeze()
            sr = 44100 # TangoFlux default
            score = calculate_clap_score(prompt, audio_np, sr)
            logs.append(f"TangoFlux Candidate {i+1}: CLAP={score:.4f}")
            if score > best_score:
                best_score = score
                best_audio = (audio_np, sr)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    out_path = f"{WORKSPACE}/gen_raw_{model_name}_{os.urandom(4).hex()}.wav"
    audio_data, sr = best_audio
    sf.write(out_path, audio_data if audio_data.ndim == 1 else audio_data.T, sr)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out_path, best_score, "\n".join(logs), model_name

# Override existing `generate_sound_effect` to use new model selection and generation
def generate_sound_effect(prompt: str, duration: float, steps: int, candidate_count: int, score_threshold: float,
                           progress=gr.Progress()):
    if not prompt.strip():
        return None, "Error: Prompt string cannot be empty.", "N/A", "N/A"
    try:
        model_to_use = select_sfx_model(prompt)
        progress(0.1, desc=f"Generating raw sound effect candidates with {model_to_use}...")
        raw_audio_path, best_score, log_summary, used_model_name = _generate_sfx_with_model(model_to_use, prompt, duration, steps, candidate_count)

        if best_score < score_threshold:
            sfx_status_msg = f"⚠️ Best SFX candidate score ({best_score:.4f}) below threshold ({score_threshold}). Kept anyway. Used model: {used_model_name}\n" + log_summary
        else:
            sfx_status_msg = f"✅ Selected best SFX candidate (CLAP: {best_score:.4f}). Used model: {used_model_name}\n" + log_summary

        progress(0.7, desc="Applying automatic mastering...")
        processed_audio_path, auto_master_explanation = auto_master(raw_audio_path, progress=progress)

        if processed_audio_path is None:
            return None, f"Auto-Mastering Error: {auto_master_explanation}\n{sfx_status_msg}", "N/A", used_model_name

        final_status_msg = sfx_status_msg + "\n\n" + auto_master_explanation
        return processed_audio_path, final_status_msg, f"{best_score:.4f}", used_model_name
    except Exception as e:
        return None, f"Generation Error: {str(e)}", "N/A", "N/A"


# Also update `generate_batch_sfx` to use the new model selection and generation
def generate_batch_sfx(prompt_lines, duration: float, steps: int, candidates: int, threshold: float, progress=gr.Progress()):
    if not prompt_lines or not prompt_lines.strip():
        return None, "Paste one prompt per line first."
    lines = [l.strip() for l in prompt_lines.splitlines() if l.strip()][:100]
    if not lines:
        return None, "No valid prompts found."

    drive_warning = ""
    if len(lines) > 20 and not DRIVE_MOUNTED:
        drive_warning = ("⚠️ Large batch without Google Drive mounted — if Colab disconnects mid-run, "
                          "unsaved progress will be lost. Consider running the optional Cell 3 (Mount Google Drive) "
                          "first, then coming back to this.\n\n")

    generated = []
    start_time = time.time()
    for i in range(len(lines)):
        p = lines[i]
        elapsed = time.time() - start_time
        avg = (elapsed / i) if i > 0 else 15.0
        remaining_min = (avg * (len(lines) - i)) / 60.0

        model_to_use = select_sfx_model(p)
        progress(i / len(lines), desc=f"{i+1}/{len(lines)} with {model_to_use}: {p[:35]}... (~{remaining_min:.1f} min left)")
        try:
            out_path, _, _, _ = _generate_sfx_with_model(model_to_use, p, duration, steps, candidates)
            processed_out_path, _ = auto_master(out_path)
            exported = _export_one(processed_out_path, "Roblox Mobile-Optimized (Mono OGG, low-mem)", out_dir=f"{WORKSPACE}/batch")
            generated.append(exported)
        except Exception as e:
            print(f"Skipped '{p}': {e}")

        if (i + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(generated, "sfx_batch")

    save_checkpoint(generated, "sfx_batch")

    if not generated:
        return None, drive_warning + "All generations failed — check the log output below the cell."

    save_note = (f"Also safely saved to Google Drive at {PERSIST_DIR}/sfx_batch_checkpoint/"
                 if DRIVE_MOUNTED else
                 "Saved to this session's temporary storage only — download it now, it won't survive a disconnect.")
    if len(generated) > 2:
        zp = zip_files(generated, "batch_sfx")
        print(f"DEBUG: Zip file path: {zp}, exists: {os.path.exists(zp)}") # Added debug print here
        return zp, f"{drive_warning}✅ Generated {len(generated)} SFX and zipped them. {save_note}"
    else:
        return generated[0], f"{drive_warning}✅ Generated {len(generated)} SFX. {save_note}"


print("✅ Cell 2 loaded: SFX engine (multi-model + Auto-Master), DSP, looping, diagnostics, export, presets, batch mode, session library.")

In [ ]:
import os

print("--- DEBUG: Contents of diffusers/pipelines/auto_pipeline.py ---")
file_path = '/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/auto_pipeline.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        file_content = f.read()
        print(file_content)
        # Also make it available as a variable in the kernel for inspection
        global auto_pipeline_content
        auto_pipeline_content = file_content
else:
    print(f"Error: File not found at {file_path}")
print("--- END DEBUG: Contents of diffusers/pipelines/auto_pipeline.py ---")

## Cell 3 — (Optional but recommended for large batches) Mount Google Drive
Only needed if you're generating a lot of assets at once — e.g. a 100-item script. Skip this for casual one-off generations. Running it means every batch job checkpoints to your Drive every 10 items, so a Colab disconnect mid-run doesn't cost you the work already done.

In [ ]:
import time as _time

def _try_mount_drive(retries=2):
    from google.colab import drive
    for attempt in range(retries):
        try:
            # force_remount clears out any stuck/partial mount from a previous
            # attempt, which is the most common cause of a plain "mount failed".
            drive.mount('/content/drive', force_remount=True)
            return True
        except Exception as e:
            print(f"Attempt {attempt+1}/{retries} failed: {e}")
            _time.sleep(2)
    return False

try:
    if _try_mount_drive():
        drive_path = "/content/drive/MyDrive/SFX_Studio_Output"
        os.makedirs(drive_path, exist_ok=True)
        set_persist_dir(drive_path, True)
        print(f"✅ Google Drive mounted. Batch outputs and checkpoints will now save to: {drive_path}")
        print("Even if this Colab session disconnects mid-batch, everything generated so far is already safe in Drive.")
    else:
        raise RuntimeError("mount failed after retries")
except Exception as e:
    print(f"⚠️ Could not mount Google Drive ({e}).")
    print("Outputs will only live in this session's temporary storage, which is WIPED if Colab disconnects or restarts.")
    print()
    print("This usually means the permission popup/link wasn't completed in time, or an old mount got stuck. Try:")
    print("  1. Runtime → Disconnect and delete runtime, then reconnect and re-run Cells 1-2, then this cell again.")
    print("  2. When the popup/link appears this time, click it immediately and finish the Google sign-in flow")
    print("     without switching tabs or letting it sit — Colab's Drive auth handshake can time out if you're slow.")
    print("  3. If it keeps failing, you can skip Drive entirely — the app still works fine, you'll just want to")
    print("     keep batches smaller (under ~20 items) and download results promptly instead of relying on checkpoints.")

## Cell 4 — Music engine + Script Analyzer
MusicGen (free, open-weights) for 1–5 min background music, and a local keyword-based script analyzer that reads a .txt file and lists every SFX/music/ambience asset it needs, then can batch-generate + checkpoint + zip them all.

## Note on Bfxr

Bfxr is a procedural sound effect *synthesizer*, not a text-to-audio generative AI model. It creates sounds based on algorithms and parameters (like 'pitch', 'waveform', 'decay') rather than directly from text descriptions. While excellent for retro/cartoony game sounds, integrating it as a text-to-audio model that picks itself based on a prompt is outside the scope of direct text-to-audio generation like the other models in this notebook. For Bfxr-style sounds, you would typically use a dedicated Bfxr tool and import the generated `.wav` files.

In [ ]:
from transformers import MusicgenForConditionalGeneration
import numpy as np

MUSICGEN_SR = 32000
MUSIC_CHUNK_SECONDS = 20  # generate in ~20s chunks, crossfade-stitch up to the target length


def load_musicgen_engine(model_name: str = "facebook/musicgen-small"):
    if ModelRegistry.musicgen_model is None or ModelRegistry.musicgen_model.config._name_or_path != model_name:
        # Free memory if a different musicgen model was loaded
        if ModelRegistry.musicgen_model is not None:
            del ModelRegistry.musicgen_model
            del ModelRegistry.musicgen_processor
            ModelRegistry.musicgen_model = None
            ModelRegistry.musicgen_processor = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        ModelRegistry.musicgen_processor = AutoProcessor.from_pretrained(model_name)
        ModelRegistry.musicgen_model = MusicgenForConditionalGeneration.from_pretrained(model_name).to(DEVICE)
        ModelRegistry.musicgen_model.eval()

def select_music_model(prompt: str):
    prompt_lower = prompt.lower()
    # Keywords suggesting higher quality or more complex music might trigger 'medium'
    if any(keyword in prompt_lower for keyword in [
        "cinematic", "epic", "orchestral", "grand", "symphonic", "rich", "detailed", "high quality"
    ]):
        # Warn if using medium on potentially constrained hardware
        print("🎵 Selecting MusicGen-Medium (higher quality, may require more GPU memory or take longer).")
        return "facebook/musicgen-medium"
    else:
        return "facebook/musicgen-small"


def _musicgen_generate_chunk(prompt, seconds, guidance_scale, temperature, music_model_name: str):
    load_musicgen_engine(music_model_name)
    inputs = ModelRegistry.musicgen_processor(text=[prompt], padding=True, return_tensors="pt").to(DEVICE)
    # ~50 tokens/sec for small, 25 tokens/sec for medium. Adjust accordingly
    if music_model_name == "facebook/musicgen-medium":
        tokens_per_sec = 25
    else:
        tokens_per_sec = 50
    max_new_tokens = int(seconds * tokens_per_sec)

    with torch.no_grad():
        audio_values = ModelRegistry.musicgen_model.generate(
            **inputs, do_sample=True, guidance_scale=guidance_scale,
            temperature=temperature, max_new_tokens=max_new_tokens
        )
    return audio_values[0, 0].cpu().numpy()


def generate_music(prompt: str, duration_minutes: float, guidance_scale: float, temperature: float,
                    progress=gr.Progress()):
    """1-5 minute music. Built by generating ~20s chunks from the same prompt and
    equal-power crossfading them together, since generating minutes of audio in one
    pass is too slow/memory-heavy for a free Colab GPU. Musical continuity between
    chunks is approximate, not a single continuous take."""
    if not prompt.strip():
        return None, "Error: Prompt cannot be empty.", "N/A"
    try:
        model_to_use = select_music_model(prompt)
        target_seconds = max(60, min(300, duration_minutes * 60))
        n_chunks = max(1, math.ceil(target_seconds / MUSIC_CHUNK_SECONDS))
        chunks = []
        for i in range(n_chunks):
            progress(i / n_chunks, desc=f"Composing section {i+1}/{n_chunks} with {model_to_use}...")
            audio_np = _musicgen_generate_chunk(prompt, MUSIC_CHUNK_SECONDS, guidance_scale, temperature, model_to_use)
            chunks.append(audio_np)

        xf_samples = int(0.5 * MUSICGEN_SR)  # 0.5s crossfade between sections
        stitched = chunks[0]
        for nxt in chunks[1:]:
            if len(stitched) > xf_samples and len(nxt) > xf_samples:
                fade_out = np.linspace(1, 0, xf_samples)
                fade_in = np.linspace(0, 1, xf_samples)
                tail = stitched[-xf_samples:] * fade_out
                head = nxt[:xf_samples] * fade_in
                crossfaded = tail + head
                stitched = np.concatenate([stitched[:-xf], crossfaded, nxt[xf_samples:]])
            else:
                stitched = np.concatenate([stitched, nxt])

        out_path = f"{WORKSPACE}/music_raw_{os.urandom(4).hex()}.wav"
        sf.write(out_path, stitched, MUSICGEN_SR)
        actual_len = len(stitched) / MUSICGEN_SR
        return out_path, f"✅ Composed {actual_len:.0f}s of music across {n_chunks} sections using {model_to_use}.", model_to_use
    except Exception as e:
        return None, f"Music Generation Error: {str(e)}", "N/A"


# ---------------- Script Analyzer ----------------

ASSET_KEYWORDS = {
    "Combat SFX": ["sword", "slash", "stab", "gunshot", "gunfire", "reload", "explosion", "blast",
                   "punch", "hit", "impact", "arrow", "shield block", "clash", "grenade", "rifle", "famas"],
    "Creature SFX": ["roar", "growl", "hiss", "screech", "snarl", "dragon", "monster", "zombie",
                     "wolf", "beast", "creature"],
    "Movement/Footsteps": ["footstep", "footsteps", "walking", "running", "jump", "land", "climb", "sprint"],
    "Environment/Ambience": ["wind", "rain", "thunder", "storm", "cave", "forest", "ocean", "river",
                             "fire crackle", "ambience", "birds", "crickets", "waterfall"],
    "Magic/Fantasy SFX": ["spell", "magic", "portal", "teleport", "enchant", "curse", "heal", "summon"],
    "UI/Interface SFX": ["button click", "menu", "notification", "ui sound", "coin", "purchase", "level up",
                          "achievement", "popup"],
    "Vehicle/Machine SFX": ["engine", "car", "vehicle", "machine", "robot", "gear", "mechanical", "drone"],
    "Doors/Objects": ["door", "chest", "lever", "switch", "creak", "unlock", "lock"],
    "Music Cue": ["battle music", "boss music", "victory theme", "main theme", "background music",
                  "calm music", "tense music", "sad theme", "ambient music", "title screen", "credits"],
    "Voice/Dialogue": ["dialogue", "voice line", "narrator", "npc says", "character says"],
}


def analyze_script(txt_file, progress=gr.Progress()):
    if txt_file is None:
        return "Please upload a .txt file.", None

    fpath = _filepath(txt_file)
    if not fpath:
        return "Could not read the uploaded file.", None
    with open(fpath, "r", errors="ignore") as f:
        text = f.read()

    lines = [l.strip() for l in re.split(r"[\n.]", text) if l.strip()]
    findings = {cat: [] for cat in ASSET_KEYWORDS}

    for line in lines:
        low = line.lower()
        for cat, keywords in ASSET_KEYWORDS.items():
            for kw in keywords:
                if kw in low:
                    snippet = line if len(line) < 90 else line[:87] + "..."
                    if snippet not in findings[cat]:
                        findings[cat].append(snippet)
                    break

    report_lines = ["# Script Audio Asset Report", ""]
    total = 0
    for cat, items in findings.items():
        if items:
            report_lines.append(f"## {cat} ({len(items)})")
            for it in items:
                report_lines.append(f"- {it}")
            report_lines.append("")
            total += len(items)

    if total == 0:
        report_lines.append("No obvious audio cues detected. Try a more descriptive script, "
                             "or generate assets manually in the SFX/Music tabs.")

    report_text = "\n".join(report_lines)
    report_path = f"{WORKSPACE}/script_asset_report_{os.urandom(3).hex()}.txt"
    with open(report_path, "w") as f:
        f.write(report_text)

    return report_text, report_path


def batch_generate_from_script(txt_file, max_items, progress=gr.Progress()):
    """Runs the analyzer, then auto-generates every detected SFX (music cues get a
    30s music stinger instead of a full track, since batch-generating multiple
    minutes-long tracks would be far too slow on a free GPU). Checkpoints to
    PERSIST_DIR every CHECKPOINT_EVERY items, and zips everything at the end if
    more than 2 files are produced."""
    if txt_file is None:
        return None, "Please upload a .txt file."

    report_text, _ = analyze_script(txt_file)
    fpath = _filepath(txt_file)
    if not fpath:
        return None, "Could not read the uploaded file."
    with open(fpath, "r", errors="ignore") as f:
        text = f.read()
    lines = [l.strip() for l in re.split(r"[\n.]", text) if l.strip()]

    to_generate = []  # (prompt, is_music)
    seen = set()
    for line in lines:
        low = line.lower()
        for cat, keywords in ASSET_KEYWORDS.items():
            for kw in keywords:
                if kw in low and line not in seen:
                    seen.add(line)
                    is_music = (cat == "Music Cue")
                    to_generate.append((line[:100], is_music))
                    break

    to_generate = to_generate[: int(max_items)]
    if not to_generate:
        return None, "No audio cues detected in this script."

    drive_warning = ""
    if len(to_generate) > 20 and not DRIVE_MOUNTED:
        drive_warning = ("⚠️ Large batch without Google Drive mounted — if Colab disconnects mid-run, "
                          "unsaved progress will be lost. Consider running the optional Cell 3 (Mount Google Drive) "
                          "first, then re-running this.\n\n")

    generated_files = []
    start_time = time.time()
    for idx, (prompt, is_music) in enumerate(to_generate):
        elapsed = time.time() - start_time
        avg = (elapsed / idx) if idx > 0 else 20.0
        remaining_min = (avg * (len(to_generate) - idx)) / 60.0
        progress(idx / len(to_generate), desc=f"{idx+1}/{len(to_generate)}: {prompt[:35]}... (~{remaining_min:.1f} min left)")
        try:
            if is_music:
                music_model_to_use = select_music_model(prompt) # Select model for music cues
                audio_np = _musicgen_generate_chunk(prompt, 30, 3.0, 1.0, music_model_to_use)
                fp = f"{WORKSPACE}/batch/music_{idx}_{os.urandom(3).hex()}.wav"
                sf.write(fp, audio_np, MUSICGEN_SR)
            else:
                sfx_model_to_use = select_sfx_model(prompt) # Use SFX model selection
                fp, _, _, _ = _generate_sfx_with_model(sfx_model_to_use, prompt, 5.0, 50, 2)
            exported = _export_one(fp, "Roblox Mobile-Optimized (Mono OGG, low-mem)", out_dir=f"{WORKSPACE}/batch")
            generated_files.append(exported)
        except Exception as e:
            print(f"Skipped '{prompt}': {e}")

        if (idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(generated_files, "script_batch")

    save_checkpoint(generated_files, "script_batch")

    if not generated_files:
        return None, drive_warning + "Generation failed for all detected items."

    save_note = (f"Also safely saved to Google Drive at {PERSIST_DIR}/script_batch_checkpoint/"
                 if DRIVE_MOUNTED else
                 "Saved to this session's temporary storage only — download it now, it won't survive a disconnect.")
    if len(generated_files) > 2:
        zip_path = zip_files(generated_files, "script_batch")
        return zip_path, f"{drive_warning}✅ Generated {len(generated_files)} assets from the script and zipped them. {save_note}"
    else:
        return generated_files[0], f"{drive_warning}✅ Generated {len(generated_files)} asset(s). {save_note}"


print("✅ Cell 3 loaded: Music engine + Script Analyzer.")


def calculate_clap_score(text_prompt: str, audio_array, sample_rate: int) -> float:
    load_clap_evaluator()
    if sample_rate != 48000:
        audio_48k = librosa.resample(audio_array, orig_sr=sample_rate, target_sr=48000)
    else:
        audio_48k = audio_array
    if audio_48k.ndim > 1:
        audio_48k = np.mean(audio_48k, axis=0)

    inputs = ModelRegistry.clap_processor(
        text=[text_prompt], audio=[audio_48k], return_tensors="pt", # Changed 'audios' to 'audio'
        sampling_rate=48000, padding=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = ModelRegistry.clap_model(**inputs)
        score = outputs.logits_per_audio[0][0].cpu().item()
    return float(score)

## Cell 5 — Video/Image → Audio Matcher engine
Uses BLIP (free, local image captioning) to describe what's in an image or video frame, turns that into an SFX prompt, generates matching audio sized to fit, and can mux it back onto the original video with ffmpeg.

In [ ]:
import subprocess
import tempfile
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
VIDEO_EXTS = {".mp4", ".mov", ".webm", ".mkv", ".avi"}


def load_blip_captioner():
    # blip_model / blip_processor slots already declared on ModelRegistry in Cell 2
    if ModelRegistry.blip_model is None:
        model_id = "Salesforce/blip-image-captioning-base"
        ModelRegistry.blip_processor = BlipProcessor.from_pretrained(model_id)
        ModelRegistry.blip_model = BlipForConditionalGeneration.from_pretrained(model_id).to(DEVICE)
        ModelRegistry.blip_model.eval()


def caption_image(image_path: str) -> str:
    load_blip_captioner()
    raw_image = Image.open(image_path).convert("RGB")
    inputs = ModelRegistry.blip_processor(raw_image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = ModelRegistry.blip_model.generate(**inputs, max_new_tokens=40)
    caption = ModelRegistry.blip_processor.decode(out[0], skip_special_tokens=True)
    return caption


def get_video_duration(video_path: str) -> float:
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", video_path],
        capture_output=True, text=True
    )
    try:
        d = float(result.stdout.strip())
        if d <= 0 or not math.isfinite(d):
            raise ValueError
        return d
    except (ValueError, TypeError):
        return 5.0  # sane fallback if ffprobe couldn't read the duration


def extract_video_frame(video_path: str, timestamp: float) -> str:
    out_path = f"{WORKSPACE}/frame_{os.urandom(3).hex()}.jpg"
    subprocess.run(
        ["ffmpeg", "-y", "-ss", str(timestamp), "-i", video_path,
         "-frames:v", "1", "-q:v", "2", out_path],
        capture_output=True
    )
    return out_path


def extend_audio_to_duration(audio_np: np.ndarray, sr: int, target_seconds: float,
                              crossfade_seconds: float = 0.3) -> np.ndarray:
    """Loop-extends a short clip up to a target length using equal-power crossfades,
    then trims to the exact length. Works well for ambience/loopable sounds; for
    one-shot impacts the tail repeats, which is expected for anything longer than
    a natural one-shot."""
    xf = int(crossfade_seconds * sr)
    target_len = int(target_seconds * sr)
    if len(audio_np) >= target_len:
        return audio_np[:target_len]
    if len(audio_np) <= xf * 2:
        # too short to crossfade meaningfully — just tile it
        reps = math.ceil(target_len / len(audio_np))
        return np.tile(audio_np, reps)[:target_len]

    stitched = audio_np.copy()
    while len(stitched) < target_len:
        fade_out = np.linspace(1, 0, xf)
        fade_in = np.linspace(0, 1, xf)
        tail = stitched[-xf:] * fade_out
        head = audio_np[:xf] * fade_in
        crossfaded = tail + head
        stitched = np.concatenate([stitched[:-xf], crossfaded, audio_np[xf:]])
    return stitched[:target_len]


def generate_matching_audio_for_prompt(prompt: str, target_duration: float):
    """SFX up to 10s generated directly; longer targets generate an 8s base
    clip and loop-extend it to fit."""
    if target_duration <= 10.0:
        out_path, best_score, _ = _generate_sfx_core(prompt, target_duration, 50, 3, 0.3)
        return out_path, best_score
    else:
        out_path, best_score, _ = _generate_sfx_core(prompt, 8.0, 50, 3, 0.3)
        y, sr = librosa.load(out_path, sr=None, mono=True)
        extended = extend_audio_to_duration(y, sr, target_duration)
        ext_path = f"{WORKSPACE}/gen_extended_{os.urandom(4).hex()}.wav"
        sf.write(ext_path, extended, sr)
        return ext_path, best_score


def mux_audio_onto_video(video_path: str, audio_path: str) -> str:
    out_path = f"{WORKSPACE}/exports/matched_video_{os.urandom(3).hex()}.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-i", audio_path,
         "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
         "-map", "0:v:0", "-map", "1:a:0", "-shortest", out_path],
        capture_output=True
    )
    return out_path


def match_audio_to_media(media_file, output_mode, custom_prompt, progress=gr.Progress()):
    """media_file: gradio File object. output_mode: 'Audio only' / 'Video with audio' / 'Both'."""
    path = _filepath(media_file)
    if path is None:
        return None, None, "Please upload an image or video file.", ""

    ext = os.path.splitext(path)[1].lower()

    try:
        if ext in IMAGE_EXTS:
            progress(0.1, desc="Looking at the image...")
            caption = custom_prompt.strip() if custom_prompt.strip() else caption_image(path)
            prompt = f"Sound effect for: {caption}"
            progress(0.4, desc="Generating matching audio...")
            audio_path, score = generate_matching_audio_for_prompt(prompt, 5.0)
            status = f"🖼️ Detected: \"{caption}\"\nGenerated matching SFX (CLAP: {score:.4f}).\nImages have no video track, so only audio is available."
            return None, audio_path, status, caption

        elif ext in VIDEO_EXTS:
            progress(0.1, desc="Reading video...")
            duration = get_video_duration(path)
            capped_duration = min(duration, 30.0)  # generation target cap for speed
            mid_ts = min(duration / 2.0, max(0, duration - 0.1))
            frame_path = extract_video_frame(path, mid_ts)

            progress(0.3, desc="Looking at the video frame...")
            caption = custom_prompt.strip() if custom_prompt.strip() else caption_image(frame_path)
            prompt = f"Sound effect for: {caption}"

            progress(0.5, desc=f"Generating {capped_duration:.0f}s of matching audio...")
            audio_path, score = generate_matching_audio_for_prompt(prompt, capped_duration)

            video_out, final_audio = None, audio_path
            if output_mode in ("Video with audio", "Both"):
                progress(0.85, desc="Muxing audio onto video...")
                # if audio shorter than video, extend once more to match exactly
                if capped_duration < duration:
                    y, sr = librosa.load(audio_path, sr=None, mono=True)
                    y_ext = extend_audio_to_duration(y, sr, duration)
                    audio_path = f"{WORKSPACE}/gen_matched_full_{os.urandom(3).hex()}.wav"
                    sf.write(audio_path, y_ext, sr)
                    final_audio = audio_path
                video_out = mux_audio_onto_video(path, audio_path)

            audio_result = final_audio if output_mode in ("Audio only", "Both") else None
            status = (f"🎬 Detected: \"{caption}\"\nVideo duration: {duration:.1f}s\n"
                      f"Generated matching audio (CLAP: {score:.4f}).")
            return video_out, audio_result, status, caption

        else:
            return None, None, f"Unsupported file type: {ext}. Upload an image (png/jpg/webp) or video (mp4/mov/webm).", ""

    except Exception as e:
        return None, None, f"Matching Error: {str(e)}", ""


print("✅ Cell 4 loaded: Video/Image → Audio Matcher engine.")


## Cell 6 — Launch the studio
Run and click the `.gradio.live` link (works on phone too).

In [ ]:
SFX_PRESETS_UI = list(SFX_PRESETS.keys())

theme = gr.themes.Soft(
    primary_hue="cyan", neutral_hue="slate"
).set(body_background_fill="#020617", block_background_fill="#0f172a", block_border_color="#1e293b")

with gr.Blocks(title="AI Audio Studio — Roblox Edition") as app:
    session_state = gr.State([])

    gr.Markdown("# 🎙️ Autonomous AI Audio Studio — Roblox Edition")
    gr.Markdown("*Free, local, mobile-safe, self-mastering. SFX, music, script-to-asset-list, and video/image audio matching.*")
    with gr.Row():
        btn_free_memory = gr.Button("🧹 Free GPU Memory", size="sm", scale=0)
        system_status = gr.Textbox(label="", placeholder="System status appears here...", interactive=False, scale=3, show_label=False)

    with gr.Tabs():
        with gr.TabItem("❓ Help"):
            gr.Markdown("""
## What each tab does

- **🔊 SFX Generator** — type a description, get a 1–10s sound effect. Generates several candidates and auto-picks the best match using CLAP scoring.
- **🎼 Music Generator** — type a description, get 1–5 minutes of background/loop music.
- **📄 Script Analyzer** — upload a .txt file (script, design doc, level notes). It scans for audio cues (combat, creatures, footsteps, ambience, magic, UI, music, etc.), lists what it finds, and can auto-generate everything in one batch.
- **🎬 Video/Image → Audio Matcher** — upload a picture or video clip, it looks at it and generates matching audio, sized to fit and optionally muxed back onto the video.
- **🎛️ DSP Mastering Rack** — polish existing audio: EQ, saturation, reverb, loudness normalization. Hit "Auto-Master" for a one-click automatic version, or tune the sliders yourself.
- **🔁 Loop Synthesis Engine** — turn a short clip into a seamless, click-free loop (good for ambience, engines, wind).
- **📊 Signal Diagnostics** — see the waveform, spectrogram, peak level, and true LUFS loudness of any clip.
- **📦 Engine Export** — convert a finished clip to the right file type for Roblox (mobile-optimized OGG), Unity/Unreal, or general use.
- **🗂️ Session Library** — every file you generate this session collects here automatically, downloadable individually or zipped.

## About Colab disconnects & "hosting 24/7"

If you saw **"reconnecting..."** mid-batch, that's Google Colab's free-tier idle/session limit — not a bug here. Free Colab sessions cap around 12 hours, and disconnect after roughly 90 minutes with no browser interaction, regardless of whether code is still running. When that happens, anything not saved outside the session (i.e. not in Google Drive) is wiped.

**The fix for big batches (built into this notebook):** run the optional **Cell 3 (Mount Google Drive)**. Every batch job now auto-checkpoints to Drive every 10 items — so a disconnect mid-run only costs you a few items, not the whole batch, and you can just re-run and keep going.

**Do you actually need "24/7 hosting"?** Probably not — you're running batch jobs (generate a pile of assets, download them, done), not a live service that needs to answer requests around the clock. True always-on GPU hosting isn't free anywhere and typically runs $0.40+/hour (roughly $300+/month) for a dedicated GPU on services like Hugging Face or a GPU rental host — overkill for occasional batch generation. Free options like Hugging Face's ZeroGPU exist but only grant a few minutes of GPU time per day, nowhere near enough for a 100-item batch. Drive-checkpointed Colab sessions are the practical free answer to what you're actually running into.
""")

        with gr.TabItem("🔊 SFX Generator (1-10s)"):
            gr.Markdown("Type a description, get a 1–10s sound effect. Several candidates are generated and the best match is auto-picked using CLAP scoring.")
            with gr.Row():
                with gr.Column(scale=2):
                    prompt_input = gr.Textbox(label="SFX Prompt", lines=3,
                        placeholder="e.g., Terrifying dragon roar echoing in a stone cavern")
                    gr.Markdown("**Quick presets:**")
                    with gr.Row():
                        preset_buttons = [gr.Button(name, size="sm") for name in list(SFX_PRESETS_UI)[:5]]
                    with gr.Row():
                        preset_buttons2 = [gr.Button(name, size="sm") for name in list(SFX_PRESETS_UI)[5:]]
                    with gr.Row():
                        dur_slider = gr.Slider(1.0, 10.0, value=3.0, step=0.5, label="Duration (1-10s)")
                        steps_slider = gr.Slider(10, 100, value=50, step=5, label="Sampling Steps")
                    with gr.Row():
                        cand_slider = gr.Slider(1, 8, value=3, step=1, label="Best-of-N Candidates")
                        thresh_slider = gr.Slider(0.1, 0.9, value=0.35, step=0.05, label="CLAP Score Threshold")
                    btn_generate = gr.Button("🔥 Generate & Evaluate", variant="primary")

                    with gr.Accordion("📋 Batch Mode — multiple prompts at once", open=False):
                        batch_prompt_box = gr.Textbox(label="One prompt per line", lines=6,
                            placeholder="Dragon roar\nSword clash\nFootsteps on gravel\nDoor creak opening")
                        btn_batch_sfx = gr.Button("🚀 Generate Batch", variant="secondary")
                        batch_sfx_status = gr.Textbox(label="Batch Status", interactive=False)
                        batch_sfx_output = gr.File(label="Batch Output (.zip if >2 files)")

                with gr.Column(scale=2):
                    status_log = gr.Textbox(label="Evaluation Log", lines=5, interactive=False)
                    score_display = gr.Textbox(label="Top CLAP Score", interactive=False)
                    gen_audio_output = gr.Audio(label="Selected Candidate", type="filepath")
                    used_sfx_model_display = gr.Textbox(label="SFX Model Used", interactive=False) # Moved here

        with gr.TabItem("🎼 Music Generator (1-5 min)"):
            gr.Markdown("Built by composing ~20s sections from the same prompt and crossfading them together — "
                        "a free GPU can't render minutes of audio in one pass, so continuity between sections "
                        "is close but approximate, not one continuous take.")
            with gr.Row():
                with gr.Column(scale=2):
                    music_prompt = gr.Textbox(label="Music Prompt", lines=3,
                        placeholder="e.g., Epic orchestral battle theme, driving percussion, heroic brass")
                    music_len = gr.Slider(1, 5, value=2, step=0.5, label="Length (minutes)")
                    with gr.Row():
                        music_guidance = gr.Slider(1.0, 6.0, value=3.0, step=0.5, label="Prompt Guidance")
                        music_temp = gr.Slider(0.5, 1.5, value=1.0, step=0.1, label="Creativity")
                    btn_music = gr.Button("🎼 Compose Music", variant="primary")
                with gr.Column(scale=2):
                    music_status = gr.Textbox(label="Status", lines=3, interactive=False)
                    music_output = gr.Audio(label="Generated Track", type="filepath")
                    used_music_model_display = gr.Textbox(label="Music Model Used", interactive=False) # Moved here

        with gr.TabItem("📄 Script Analyzer"):
            gr.Markdown("Upload a .txt script, design doc, or level description. It scans for combat, creature, "
                        "movement, ambience, magic, UI, vehicle, and music cues and lists every asset you'll need. "
                        "You can then batch-generate all of them at once, mobile-optimized and zipped automatically "
                        "when there are more than 2 files.")
            with gr.Row():
                with gr.Column(scale=1):
                    script_file = gr.File(label="Upload .txt file", file_types=[".txt"])
                    btn_analyze = gr.Button("📄 Analyze Script", variant="secondary")
                    max_items_slider = gr.Slider(1, 150, value=15, step=1, label="Max assets to auto-generate")
                    gr.Markdown("⏱️ Rough pace: ~15-25s per SFX, ~60-90s per music cue. A 100-item batch can take 30-60+ min — "
                                "mount Google Drive (Cell 3) first so nothing is lost if the session disconnects.")
                    btn_batch = gr.Button("🚀 Generate All Detected Assets", variant="primary")
                with gr.Column(scale=2):
                    report_box = gr.Textbox(label="Detected Asset Report", lines=16, interactive=False)
                    report_file = gr.File(label="Download Report (.txt)")
                    batch_status = gr.Textbox(label="Batch Generation Status", interactive=False)
                    batch_output = gr.File(label="Download Generated Assets (.zip if >2 files)")

        with gr.TabItem("🎬 Video/Image → Audio Matcher"):
            gr.Markdown("Upload an image or video. It looks at the content (BLIP image captioning, free & local), "
                        "writes a matching sound description, and generates audio sized to fit — a still image gets "
                        "a 5s SFX, a video gets audio matched to its length (up to 30s of generated content, then "
                        "loop-extended to cover the rest). Choose whether you want just the audio, the video with "
                        "the new audio muxed on, or both.")
            with gr.Row():
                with gr.Column(scale=1):
                    media_upload = gr.File(label="Upload Image or Video",
                        file_types=["image", "video", ".png", ".jpg", ".jpeg", ".webp",
                                    ".mp4", ".mov", ".webm", ".mkv"])
                    media_custom_prompt = gr.Textbox(label="Override description (optional)", lines=2,
                        placeholder="Leave blank to let the AI describe it automatically")
                    media_output_mode = gr.Radio(
                        choices=["Audio only", "Video with audio", "Both"],
                        value="Both", label="What do you want back?")
                    btn_match = gr.Button("🎬 Analyze & Generate Matching Audio", variant="primary")
                with gr.Column(scale=2):
                    match_status = gr.Textbox(label="Status", lines=4, interactive=False)
                    detected_caption = gr.Textbox(label="Detected Description", interactive=False)
                    match_video_output = gr.Video(label="Video with Matched Audio")
                    match_audio_output = gr.Audio(label="Matched Audio Only", type="filepath")

        with gr.TabItem("🎛️ DSP Mastering Rack"):
            gr.Markdown("Use **Auto-Master** for a one-click result based on real analysis of your audio, or tune the rack manually below.")
            with gr.Row():
                with gr.Column(scale=1):
                    dsp_source_audio = gr.Audio(label="Source Input Audio", type="filepath")
                    btn_auto_master = gr.Button("🧠 Auto-Master (one click)", variant="primary")
                    gr.Markdown("**— or tune manually —**")
                    hp_filter_s = gr.Slider(20, 500, value=80, step=10, label="High-Pass Cutoff (Hz)")
                    eq_mid_s = gr.Slider(-12, 12, value=2, step=1, label="Mid Boost (dB @ 2kHz)")
                    high_shelf_s = gr.Slider(-12, 12, value=0, step=1, label="High Shelf Gain (dB @ 8kHz)") # Added
                    drive_s = gr.Slider(0.0, 1.0, value=0.15, step=0.05, label="Saturation Drive")
                    reverb_s = gr.Slider(0.0, 1.0, value=0.2, step=0.05, label="Comb Reverb Mix")
                    noise_gate_s = gr.Slider(-90, -30, value=-60, step=5, label="Noise Gate Threshold (dB)") # Added
                    comp_threshold_s = gr.Slider(-30, 0, value=0, step=1, label="Compressor Threshold (dB)") # Added
                    comp_ratio_s = gr.Slider(1.0, 10.0, value=1.0, step=0.5, label="Compressor Ratio") # Added
                    limiter_threshold_s = gr.Slider(-5.0, 0.0, value=-0.5, step=0.1, label="Limiter Threshold (dB)") # Added
                    lufs_chk = gr.Checkbox(value=True, label="Normalize Loudness (-14 LUFS / -0.5dB Peak, true ITU-R BS.1770 metering)")
                    btn_process_dsp = gr.Button("⚙️ Apply Manual DSP Chain", variant="secondary")
                with gr.Column(scale=2):
                    dsp_status = gr.Textbox(label="DSP Log", lines=6, interactive=False)
                    dsp_audio_output = gr.Audio(label="Processed Audio Output", type="filepath")

        with gr.TabItem("🔁 Loop Synthesis Engine"):
            gr.Markdown("Turn a short clip into a seamless, click-free loop — good for ambience, engines, wind, and other continuous sounds.")
            with gr.Row():
                with gr.Column():
                    loop_source = gr.Audio(label="Source Audio to Loop", type="filepath")
                    xf_len_s = gr.Slider(50, 1000, value=250, step=10, label="Equal-Power Crossfade (ms)")
                    btn_loop = gr.Button("🔁 Generate Seamless Loop", variant="primary")
                with gr.Column():
                    loop_status = gr.Textbox(label="Looping Status", interactive=False)
                    loop_audio_output = gr.Audio(label="Continuous Loop Preview (3x Loop)", type="filepath")

        with gr.TabItem("📊 Signal Diagnostics"):
            gr.Markdown("See the waveform, spectrogram, peak level, and true LUFS loudness (ITU-R BS.1770 metering) of any clip.")
            with gr.Row():
                with gr.Column(scale=1):
                    diag_source = gr.Audio(label="Audio for Signal Diagnostics", type="filepath")
                    btn_diag = gr.Button("📊 Run Diagnostics", variant="primary")
                    diag_metrics = gr.Textbox(label="Calculated Metrics", lines=6, interactive=False)
                with gr.Column(scale=2):
                    diag_plot = gr.Image(label="Waveform & Spectrogram Plot")

        with gr.TabItem("📦 Engine Export"):
            gr.Markdown("**For Roblox 3D positional audio:** always export **mono**. Roblox pans/attenuates sound "
                        "itself in 3D via the Sound object's position — a stereo file breaks that positioning. "
                        "Use **Mobile-Optimized** for anything that plays a lot in-game (footsteps, ambience loops, "
                        "gunfire) to keep phone memory/CPU usage low; use full-quality OGG for rare, high-impact "
                        "one-shots (boss roars, cutscene stings).")
            with gr.Row():
                with gr.Column():
                    export_source = gr.Audio(label="Audio Asset to Export", type="filepath")
                    preset_dropdown = gr.Dropdown(
                        choices=[
                            "Roblox Mobile-Optimized (Mono OGG, low-mem)",
                            "Roblox Mono OGG (44.1kHz, full quality)",
                            "Unreal/Unity Stereo WAV (48kHz)",
                            "High-Quality Lossless FLAC",
                            "Lightweight Web MP3",
                        ],
                        value="Roblox Mobile-Optimized (Mono OGG, low-mem)",
                        label="Target Export Profile")
                    btn_export = gr.Button("🚀 Export Master Asset", variant="primary")
                with gr.Column():
                    export_status = gr.Textbox(label="Export Log", interactive=False)
                    export_file_out = gr.File(label="Download Formatted File")

        with gr.TabItem("🗂️ Session Library"):
            gr.Markdown("Everything you generate in this session (SFX, batches, music, script batches, matched video/audio) "
                        "collects here automatically. Download files individually below, or zip the whole session "
                        "(kicks in once you have more than 2 files, same rule as everywhere else in the app).")
            session_gallery = gr.Files(label="Everything generated this session", interactive=False)
            with gr.Row():
                btn_zip_session = gr.Button("📦 Zip Session Library", variant="primary")
                btn_clear_session = gr.Button("🗑️ Clear Library List", variant="secondary")
            session_zip_status = gr.Textbox(label="Status", interactive=False)
            session_zip_output = gr.File(label="Session Zip")

    # ---- Wiring ----

    btn_free_memory.click(free_gpu_memory, outputs=[system_status])

    for btn, name in zip(preset_buttons + preset_buttons2, list(SFX_PRESETS_UI)):
        btn.click(fn=(lambda n=name: apply_preset(n)), outputs=[prompt_input])

    btn_generate.click(generate_sound_effect,
        inputs=[prompt_input, dur_slider, steps_slider, cand_slider, thresh_slider],
        outputs=[gen_audio_output, status_log, score_display, used_sfx_model_display] # Added used_sfx_model_display
    ).then(add_to_library, [gen_audio_output, session_state], [session_state, session_gallery])

    btn_batch_sfx.click(generate_batch_sfx,
        inputs=[batch_prompt_box, dur_slider, steps_slider, cand_slider, thresh_slider],
        outputs=[batch_sfx_output, batch_sfx_status]
    ).then(add_to_library, [batch_sfx_output, session_state], [session_state, session_gallery])

    btn_music.click(generate_music,
        inputs=[music_prompt, music_len, music_guidance, music_temp],
        outputs=[music_output, music_status, used_music_model_display] # Added used_music_model_display
    ).then(add_to_library, [music_output, session_state], [session_state, session_gallery])

    btn_analyze.click(analyze_script, inputs=[script_file], outputs=[report_box, report_file])
    btn_batch.click(batch_generate_from_script, inputs=[script_file, max_items_slider],
        outputs=[batch_output, batch_status]
    ).then(add_to_library, [batch_output, session_state], [session_state, session_gallery])

    btn_match.click(match_audio_to_media,
        inputs=[media_upload, media_output_mode, media_custom_prompt],
        outputs=[match_video_output, match_audio_output, match_status, detected_caption]
    ).then(add_to_library, [match_video_output, session_state], [session_state, session_gallery]
    ).then(add_to_library, [match_audio_output, session_state], [session_state, session_gallery])

    btn_auto_master.click(auto_master, inputs=[dsp_source_audio], outputs=[dsp_audio_output, dsp_status]
    ).then(add_to_library, [dsp_audio_output, session_state], [session_state, session_gallery])

    btn_process_dsp.click(process_dsp_chain,
        inputs=[dsp_source_audio, hp_filter_s, eq_mid_s, drive_s, reverb_s, lufs_chk, # Existing
                noise_gate_s, high_shelf_s, comp_threshold_s, comp_ratio_s, limiter_threshold_s], # Added
        outputs=[dsp_audio_output, dsp_status]
    ).then(add_to_library, [dsp_audio_output, session_state], [session_state, session_gallery])

    btn_loop.click(generate_seamless_loop, inputs=[loop_source, xf_len_s], outputs=[loop_audio_output, loop_status]
    ).then(add_to_library, [loop_audio_output, session_state], [session_state, session_gallery])

    btn_diag.click(analyze_signal_metrics, inputs=[diag_source], outputs=[diag_metrics, diag_plot])

    btn_export.click(export_game_preset, inputs=[export_source, preset_dropdown],
        outputs=[export_file_out, export_status]
    ).then(add_to_library, [export_file_out, session_state], [session_state, session_gallery])

    btn_zip_session.click(zip_session_library, inputs=[session_state], outputs=[session_zip_output, session_zip_status])
    btn_clear_session.click(clear_session_library, outputs=[session_state, session_gallery])

app.queue().launch(share=True, debug=True, theme=theme)


In [ ]:
# Deprecated: `used_sfx_model_display` and `used_music_model_display` moved to `cell_id: Dj0Yxmuw7krx`

In [ ]:
# Deprecated: Wiring for `used_sfx_model_display` and `used_music_model_display` moved to `cell_id: Dj0Yxmuw7krx`